# V7_A_N06 — Seasonal Crop Production Forecasting

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft. All data are synthetic and illustrative. Analytical outputs require review by the named authority.

## Decision contract
**Decision:** plan supply, reserves, imports, market preparedness, and field verification. **Owners:** agriculture and national statistical authorities. **Horizon:** seasonal. **Boundary:** forecasts complement—not replace—official estimates and expert crop assessment.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(7606)
years=np.arange(2000,2027); rainfall=rng.normal(520,65,len(years)); area=180+2.8*(years-2000)+rng.normal(0,8,len(years)); yield_t=1.6+.0025*rainfall+rng.normal(0,.16,len(years)); production=area*yield_t
df=pd.DataFrame({'year':years,'rainfall_mm':rainfall,'area_kha':area,'yield_t_ha':yield_t,'production_kt':production})
df.tail().round(2)

## Evidence contract
Production equals harvested area × yield when concepts and units align. Rainfall is an explanatory signal, not a substitute for crop-cutting, area measurement, or official reconciliation.

In [2]:
assert df.year.is_unique and df[['area_kha','yield_t_ha','production_kt']].gt(0).all().all()
identity_error=(df.production_kt-df.area_kha*df.yield_t_ha).abs().max()
print('IDENTITY_MAX_ERROR',identity_error,'YEARS',len(df))

IDENTITY_MAX_ERROR 0.0 YEARS 27


## Time-ordered holdout and transparent baselines
The last five seasons are held out. We compare last observation, historical mean, and linear trend before adding climate information.

In [3]:
train=df.iloc[:-5].copy(); test=df.iloc[-5:].copy(); test['last']=train.production_kt.iloc[-1]; test['mean']=train.production_kt.mean(); trend=np.polyfit(train.year,train.production_kt,1); test['trend']=np.polyval(trend,test.year)
def metric(a,p):
 e=np.asarray(a)-np.asarray(p); return {'MAE':np.abs(e).mean(),'RMSE':np.sqrt((e**2).mean()),'Bias':e.mean()}
scores=pd.DataFrame({c:metric(test.production_kt,test[c]) for c in ['last','mean','trend']}).T
print(scores.round(2).to_string())

          MAE    RMSE    Bias
last    44.14   51.96   19.97
mean   118.38  127.73  118.38
trend   45.96   53.76   12.56


## Multivariate candidate
A transparent regression combines year, rainfall, and area. It must outperform relevant baselines on future seasons and remain scientifically interpretable.

In [4]:
X=lambda d:np.column_stack([np.ones(len(d)),d.year-df.year.min(),d.rainfall_mm,d.area_kha])
beta=np.linalg.lstsq(X(train),train.production_kt,rcond=None)[0]; test['multivariate']=X(test)@beta; scores.loc['multivariate']=metric(test.production_kt,test.multivariate)
print(scores.round(2).sort_values('MAE').to_string())

                 MAE    RMSE    Bias
multivariate   32.47   40.06   -8.60
last           44.14   51.96   19.97
trend          45.96   53.76   12.56
mean          118.38  127.73  118.38


## Uncertainty and calibration
Use historical one-step-like residuals as an illustrative calibration set. Operational work needs rolling-origin evaluation and interval coverage by crop, region, and horizon.

In [5]:
fitted=X(train)@beta; q=np.quantile(np.abs(train.production_kt-fitted),.90); test['lower']=test.multivariate-q; test['upper']=test.multivariate+q; coverage=((test.production_kt>=test.lower)&(test.production_kt<=test.upper)).mean()
print('INTERVAL_HALF_WIDTH',round(q,2),'HOLDOUT_COVERAGE',round(coverage,2))

INTERVAL_HALF_WIDTH 33.06 HOLDOUT_COVERAGE 0.4


## Scenario analysis is not a prediction
Decision-makers may ask what happens under rainfall or area assumptions. Label these as scenarios and preserve the assumption set.

In [6]:
base={'year':2027,'rainfall_mm':520,'area_kha':260}; scenarios=[]
for label,rain,area in [('dry',420,250),('reference',520,260),('wet',610,265)]:
 row=pd.DataFrame([{'year':2027,'rainfall_mm':rain,'area_kha':area}]); pred=(X(row)@beta).item(); scenarios.append({'scenario':label,'rainfall_mm':rain,'area_kha':area,'production_kt':round(pred,1)})
print(pd.DataFrame(scenarios).to_string(index=False))

 scenario  rainfall_mm  area_kha  production_kt
      dry          420       250          684.8
reference          520       260          768.1
      wet          610       265          828.6


## Decision product and abstention
The briefing output includes the baseline comparison, interval, scenario assumptions, source vintage, and reasons to abstain.

In [7]:
best=scores.MAE.idxmin(); product={'selected_method':best,'holdout_MAE':round(float(scores.loc[best,'MAE']),2),'interval_note':'illustrative; rolling-origin calibration required','official_estimate_status':'NOT REPLACED','abstain_when':['area estimate unavailable','rainfall coverage inadequate','concept/unit break','unreconciled shock']}
print(product)

{'selected_method': 'multivariate', 'holdout_MAE': 32.47, 'interval_note': 'illustrative; rolling-origin calibration required', 'official_estimate_status': 'NOT REPLACED', 'abstain_when': ['area estimate unavailable', 'rainfall coverage inadequate', 'concept/unit break', 'unreconciled shock']}


## Exercises
1. Implement rolling-origin validation. 2. Compare yield-first and direct-production models. 3. Add a drought-shock stress test. 4. Explain why import quantities cannot be derived mechanically from the point forecast.

## Exact solutions
1. At each historical origin, train only on prior seasons and score the next defined horizon. 2. Yield-first preserves area × yield structure; direct models may fit production but obscure components—compare both out of time. 3. Apply plausible rainfall/yield shocks and report error/interval degradation. 4. Trade decisions also require stocks, consumption, losses, prices, logistics, policy objectives, uncertainty, and authority.

In [8]:
assert test[['last','mean','trend','multivariate','lower','upper']].notna().all().all()
assert product['official_estimate_status']=='NOT REPLACED'
print('V7_A_N06_REWORK_COMPLETE_EXECUTION_PASS')

V7_A_N06_REWORK_COMPLETE_EXECUTION_PASS
